# Comparativo de Resultados - Benchmarks do Logscan

Este notebook lê os arquivos gerados pelo Logscan (que começam com `benchmark_`) e apresenta gráficos comparativos de Parsing Accuracy (PA), FTA, GA e FGA separados por dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

csv_files = glob.glob('benchmark_*.csv')

metrics = ['Accuracy', 'FTA', 'GA', 'FGA']
data_dict_no_test = {m: {} for m in metrics}
data_dict_test = {m: {} for m in metrics}

for file in csv_files:
    filename = os.path.basename(file)
    label = filename.replace('benchmark_', '').replace('.csv', '')
    try:
        df = pd.read_csv(file, index_col='Dataset')
        for metric in metrics:
            if metric in df.columns:
                if 'test' in label.lower():
                    data_dict_test[metric][label] = df[metric]
                else:
                    data_dict_no_test[metric][label] = df[metric]
    except Exception as e:
        print(f"Ignorando {file}: {e}")


In [ ]:
def plot_metrics(data_dict, title_suffix):
    for metric in metrics:
        if not data_dict[metric]:
            continue
        df_plot = pd.DataFrame(data_dict[metric])
        if df_plot.empty:
            continue
        cols = list(df_plot.columns)
        if 'LILAC' in cols:
            cols.remove('LILAC')
            cols.append('LILAC')
            df_plot = df_plot[cols]

        ax = df_plot.plot(kind='bar', figsize=(16, 6), width=0.8, colormap='tab10')
        metric_name = 'Parsing Accuracy' if metric == 'Accuracy' else metric

        plt.title(f'Comparativo de {metric_name} {title_suffix}', fontsize=16, fontweight='bold')
        plt.ylabel(metric_name, fontsize=14)
        plt.xlabel('Dataset', fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.ylim([0, 1.05])
        plt.legend(title='Versão / Configuração', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

plot_metrics(data_dict_no_test, "")


In [ ]:
plot_metrics(data_dict_test, "(Test run)")
